In [16]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import pickle
from torch.utils.data import Dataset

from language_model.tokenizer import Tokenizer
from language_model.text_generator import TextGenerator

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

In [15]:
import os
#print(os.getcwd())
os.chdir("..\\Transformer_from_Scratch")
print(os.getcwd())

C:\Personal Projects\Transformer_from_Scratch


In [17]:
# Training Tokenizer
data = "data/my_essays_data.txt"

tokenizer = Tokenizer(vocab_size=100_000)
tokenizer.fit([data])

with open("models/generation_tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [20]:
class TextDataset(Dataset):
    def __init__(self, tokens, seq_len, stride):
        self.tokens = tokens
        self.seq_len = seq_len
        self.stride = stride

    def __len__(self):
        return (len(self.tokens) - self.seq_len) // self.stride
    
    def __getitem__(self, idx):

        start = idx * self.stride

        x = self.tokens[start:start + self.seq_len]
        y = self.tokens[start + 1:start + self.seq_len + 1]
       
        return torch.tensor(x), torch.tensor(y)

In [24]:
# Training the model
model = TextGenerator(
    vocab_size=tokenizer.vocab_size,
    embedding_dim=512,
    num_layers=6,
    num_heads=8,
    d_ff=2048,
    max_len=100,
    dropout=0.1,
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
)

dataset = TextDataset(tokenizer.transform(data), seq_len=100, stride=1)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=3e-4)
batch_size = 64
epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for x, y in dataloader:
        x = x.to(model.device)
        y = y.to(model.device)

        trg_mask = model.make_causal_mask(x.shape[1])

        output = model(x, trg_mask=trg_mask)
        loss = criterion(output.reshape(-1, output.shape[-1]), y.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

ValueError: __len__() should return >= 0